# Phase 1d — Hybrid Neural Classifier and Imbalance Study

Runs the full six-arm study on a Colab GPU and hands back the trained weights.

**Before you start:** `Runtime → Change runtime type → GPU`.

Rebuilds everything from source — downloads MIT-BIH from PhysioNet and re-runs
preprocessing — so there is nothing to upload and no chance of training against
a stale copy of the beats. Roughly 30–60 minutes depending on the GPU.

**Cells 4, 5b and 5d are gates.** They raise and stop the notebook if something
is wrong. PhysioNet drops connections fairly often, and an incomplete download
produces a study that trains on 47 records and scores a 21-record DS2 — numbers
that look fine and are silently not comparable to the Phase 1b baseline.


## 1. Environment and GPU

`CUBLAS_WORKSPACE_CONFIG` is set **first**, before anything imports torch: CUDA
reads it when it creates its context, and setting it later has no effect. It is
what makes deterministic matmul kernels available.


In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['SMART_HEALTH_DATA'] = '/content/physionet-data'

!nvidia-smi -L || echo 'NO GPU -- Runtime -> Change runtime type -> GPU'

import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('No GPU. Change the runtime type before continuing.')


## 2. Upload the code

Build the archive on your laptop first:

```
./venv/bin/python scripts/package_for_colab.py
```

That writes `phase1d-colab.zip` (~122 KB). Upload it when prompted.


In [ ]:
from google.colab import files
import zipfile, pathlib, shutil

uploaded = files.upload()
name = next(iter(uploaded))

work = pathlib.Path('/content/phase1d')
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True)

with zipfile.ZipFile(name) as archive:
    archive.extractall(work)

%cd /content/phase1d
!ls


## 3. Install dependencies

`torch` is deliberately not reinstalled: Colab ships a CUDA build, and
overwriting it risks losing GPU support. Whatever version is present is
recorded in the run's provenance block.


In [ ]:
!pip install -q -r requirements-colab.txt
import numpy, scipy, sklearn, wfdb
print('numpy', numpy.__version__, '| scipy', scipy.__version__,
      '| sklearn', sklearn.__version__, '| wfdb', wfdb.__version__)


## 4. GATE — run the test suite before spending GPU time

303 tests, about 10 seconds. If any fail, the code did not survive the trip;
fix that before training rather than after.


In [ ]:
import subprocess
result = subprocess.run(['python', '-m', 'pytest', '-q'], text=True)
if result.returncode != 0:
    raise SystemExit('Tests failed. Do not train until this is green.')
print('\ntests green')


## 5a. Download MIT-BIH

48 records, 93.9 MB. The script is idempotent — records already present and
verified are skipped — so re-running this cell costs nothing and repairs a
partial download. It is retried automatically, because PhysioNet drops
connections often enough to matter.


In [ ]:
import subprocess

for attempt in range(1, 6):
    print(f'--- download attempt {attempt} ---', flush=True)
    subprocess.run(['python', 'scripts/download_mitbih.py'], text=True)
    check = subprocess.run(
        ['python', 'scripts/download_mitbih.py', '--verify'],
        text=True, capture_output=True,
    )
    print(check.stdout.strip().splitlines()[-1] if check.stdout else '')
    if check.returncode == 0:
        print('all records verify')
        break
    print('incomplete, retrying the missing records...', flush=True)
else:
    raise SystemExit('Download still incomplete after 5 attempts.')


## 5b. GATE — all 48 records present and intact

A missing record silently shrinks DS2 and makes every number in the study
incomparable to the Phase 1b baseline. This refuses to continue.


In [ ]:
import subprocess, sys
sys.path.insert(0, 'src')
from ecg import config

verify = subprocess.run(
    ['python', 'scripts/download_mitbih.py', '--verify'], text=True
)
if verify.returncode != 0:
    raise SystemExit('Records missing or corrupt. Re-run cell 5a.')
print(f'all {len(config.ALL_RECORDS)} records verify')


## 5c. Preprocess


In [ ]:
!python scripts/preprocess_mitbih.py \
    --strategy ds1_ds2 --out-dir /content/physionet-data/processed/mitdb-ds12


## 5d. GATE — the processed data matches the Phase 1b baseline exactly

Checks the split sizes and beat counts against what Phase 1b actually used.
If these differ, the study is not comparable to the published floor and there
is no point training.


In [ ]:
import json, pathlib, sys
sys.path.insert(0, 'src')
from ecg import config, dataset, splits

out = pathlib.Path('/content/physionet-data/processed/mitdb-ds12')
beats = out / 'beats'

n_npz = len(list(beats.glob('*.npz')))
assert n_npz == 48, f'expected 48 processed records, found {n_npz}'

assignment = json.loads((out / 'splits.json').read_text())
assert len(assignment['test']) == 22, (
    f"DS2 has {len(assignment['test'])} records, expected 22 -- "
    f'a record is missing and the results would not be comparable'
)

ds1 = dataset.load_records(beats, splits.DS1_RECORDS)
ds2 = dataset.load_records(beats, splits.DS2_RECORDS)
counts = dataset.class_counts(ds2)

print(f'DS1 {ds1.n_kept:,} beats   DS2 {ds2.n_kept:,} beats')
print('DS2 class counts:', dict(zip(config.AAMI_CLASSES, counts.tolist())))

assert ds2.n_kept == 49646, f'DS2 has {ds2.n_kept} beats, Phase 1b used 49,646'
print('\nmatches the Phase 1b baseline exactly')


## 6. Check determinism on this GPU

Verified locally as bit-identical on CPU and MPS. CUDA uses different kernels,
so it is checked here rather than assumed. A previous run on a T4 differed by
4.3e-4 without the flags now set in cell 1.

If it still is not deterministic, the study can run — but the write-up has to
say so rather than claim reproducibility it does not have.


In [ ]:
import sys
sys.path.insert(0, 'src'); sys.path.insert(0, 'tests')
import numpy as np
from test_train import make_data
from ecg.nn import train

cfg = train.TrainConfig(epochs=2, batch_size=64, seed=42, device='cuda')
a = train.train_model(make_data(n=600), make_data(n=300, seed=1), 'weighted', cfg)
b = train.train_model(make_data(n=600), make_data(n=300, seed=1), 'weighted', cfg)

identical = np.array_equal(a.val_proba, b.val_proba)
drift = float(np.abs(a.val_proba - b.val_proba).max())
print('CUDA deterministic across runs:', identical)
print('max abs difference:', drift)

DETERMINISTIC = identical
if not identical:
    print(f'\nStill non-deterministic (drift {drift:.2e}). This is a known CUDA\n'
          'behaviour, not a bug in the code. The study may proceed, but record\n'
          'this in the write-up -- the numbers are not exactly reproducible.')


## 7. Run the study

40 training runs: a weight-cap sweep (10/25/50), five arms across five
record-grouped folds, then five final fits and **one** pass over DS2.

`nn_cv.json` is written after every arm, so a dropped session loses only the
arm in flight.


In [ ]:
!python scripts/train_nn.py \
    --beats-dir /content/physionet-data/processed/mitdb-ds12/beats \
    --out-dir   /content/physionet-data/processed/mitdb-ds12 \
    --device cuda --epochs 40 \
    2>&1 | tee /content/phase1d-run.log


## 8. Collect the results and weights

Six checkpoints at ~250 KB each, plus the JSON and the log — under 2 MB.


In [ ]:
import shutil, pathlib, json

out = pathlib.Path('/content/physionet-data/processed/mitdb-ds12')
bundle = pathlib.Path('/content/phase1d-results')
if bundle.exists():
    shutil.rmtree(bundle)
(bundle / 'checkpoints').mkdir(parents=True)

for name in ('nn_cv.json', 'nn_metrics.json', 'nn_folds.json'):
    src = out / name
    if src.exists():
        shutil.copy(src, bundle / name)
        print('collected', name)
    else:
        print('MISSING', name)

for ckpt in sorted((out / 'checkpoints').glob('*.pt')):
    shutil.copy(ckpt, bundle / 'checkpoints' / ckpt.name)
    print('collected checkpoints/' + ckpt.name)

shutil.copy('/content/phase1d-run.log', bundle / 'run.log')

# Record whether this GPU was deterministic, so the write-up cannot forget.
(bundle / 'determinism.json').write_text(json.dumps({
    'cuda_deterministic': bool(DETERMINISTIC),
    'gpu': torch.cuda.get_device_name(0),
    'torch': torch.__version__,
}, indent=2) + '\n')

shutil.make_archive('/content/phase1d-results', 'zip', bundle)
size = pathlib.Path('/content/phase1d-results.zip').stat().st_size / 1e6
print(f'\nbundle: {size:.2f} MB')


### Save to Drive (survives the session ending)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
target = '/content/drive/MyDrive/phase1d-results.zip'
shutil.copy('/content/phase1d-results.zip', target)
print('saved to', target)


### Or download straight to your laptop


In [ ]:
from google.colab import files
files.download('/content/phase1d-results.zip')


## 9. Back on your laptop

```
./venv/bin/python scripts/ingest_colab_results.py ~/Downloads/phase1d-results.zip
```

Copies the metrics into `results/`, the weights into `models/`, and prints the
comparison table.
